# Smart Traffic Intelligence System

## Phase 4: Accident Hotspot Detection Using DBSCAN

### Objective
To identify geographical areas with a high concentration of road accidents using the DBSCAN clustering algorithm.

### Algorithm
DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

### Input Features
- Latitude
- Longitude

### Expected Outputs
1. Geographical accident clusters
2. Number of identified hotspots
3. Noise/outlier accident records
4. Accident distribution by cluster
5. Interactive hotspot map
6. Exported hotspot dataset

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium

from pathlib import Path
from sklearn.cluster import DBSCAN
from IPython.display import display
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

sns.set_theme(style="whitegrid")

print("All libraries imported successfully!")

All libraries imported successfully!


In [14]:
PROJECT_DIR = Path.cwd()

# If notebook is running from the notebooks folder,
# move one level up to the project directory.
if not (PROJECT_DIR / "dataset").exists():
    PROJECT_DIR = PROJECT_DIR.parent

RAW_DATA_PATH = PROJECT_DIR / "dataset" / "raw" / "traffic_accidents.csv"

PROCESSED_DIR = PROJECT_DIR / "dataset" / "processed"
MAP_DIR = PROJECT_DIR / "visualizations" / "maps"
REPORT_DIR = PROJECT_DIR / "visualizations" / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Dataset path:", RAW_DATA_PATH)
print("Dataset available:", RAW_DATA_PATH.exists())

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        "traffic_accidents.csv was not found. "
        "Place it inside dataset/raw/"
    )

Project directory: c:\Users\Pavan Naik\OneDrive\Desktop\DSV-project
Dataset path: c:\Users\Pavan Naik\OneDrive\Desktop\DSV-project\dataset\raw\traffic_accidents.csv
Dataset available: True


In [15]:
df = pd.read_csv(
    RAW_DATA_PATH,
    low_memory=False
)

# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

print("=" * 55)
print("ACCIDENT DATASET LOADED")
print("=" * 55)

print("Total records:", len(df))
print("Total columns:", len(df.columns))

print("\nAvailable columns:")
print(df.columns.tolist())

display(df.head())

ACCIDENT DATASET LOADED
Total records: 2301644
Total columns: 29

Available columns:
['crash_date', 'crash_time', 'borough', 'zip_code', 'latitude', 'longitude', 'location', 'on_street_name', 'cross_street_name', 'off_street_name', 'number_of_persons_injured', 'number_of_persons_killed', 'number_of_pedestrians_injured', 'number_of_pedestrians_killed', 'number_of_cyclist_injured', 'number_of_cyclist_killed', 'number_of_motorist_injured', 'number_of_motorist_killed', 'contributing_factor_vehicle_1', 'contributing_factor_vehicle_2', 'contributing_factor_vehicle_3', 'contributing_factor_vehicle_4', 'contributing_factor_vehicle_5', 'collision_id', 'vehicle_type_code_1', 'vehicle_type_code_2', 'vehicle_type_code_3', 'vehicle_type_code_4', 'vehicle_type_code_5']


,crash_date,crash_time,borough,zip_code,latitude,longitude,location,on_street_name,cross_street_name,off_street_name,number_of_persons_injured,number_of_persons_killed,number_of_pedestrians_injured,number_of_pedestrians_killed,number_of_cyclist_injured,number_of_cyclist_killed,number_of_motorist_injured,number_of_motorist_killed,contributing_factor_vehicle_1,contributing_factor_vehicle_2,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,collision_id,vehicle_type_code_1,vehicle_type_code_2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5
0,10/27/2026,15:48,MANHATTAN,10013.0,40.720448,-74.005590,"(40.720448, -74.00559)",NaN,260 W BROADWAY,NaN,0.0,0.0,0,0,0,0,0,0,Does Not Apply,Does Not Apply,NaN,NaN,NaN,73fa239b448001d1b74e285a42d70488,4 dr sedan,Taxi,NaN,NaN,NaN
1,10/27/2026,23:54,BROOKLYN,11219.0,40.641815,-73.995810,"(40.641815, -73.99581)",10 AVE,NaN,44 ST,0.0,0.0,0,0,0,0,0,0,Does Not Apply,Does Not Apply,NaN,NaN,NaN,1ca9703d3d16fae179d917b36065331a,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN
2,10/27/2026,12:35,QUEENS,11433.0,40.704185,-73.785980,"(40.704185, -73.78598)",172 ST,NaN,LIBERTY AVE,2.0,0.0,0,0,0,0,2,0,Driver Inattention/Distraction,Driver Inattention/Distraction,NaN,NaN,NaN,005e53cae38738ee5bbedf0da83662f1,4 dr sedan,4 dr sedan,NaN,NaN,NaN
3,10/27/2026,07:35,BRONX,NaN,NaN,NaN,NaN,W 179TH,NaN,MAJOR DEEGAN EXPY,0.0,0.0,0,0,0,0,0,0,Does Not Apply,Does Not Apply,NaN,NaN,NaN,ca38ea795663bff25cf2728441997fbe,4 dr sedan,NaN,NaN,NaN,NaN
4,10/27/2026,15:53,MANHATTAN,10034.0,40.860910,-73.921524,"(40.86091, -73.921524)",ACADEMY ST,NaN,10 AVE,0.0,0.0,0,0,0,0,0,0,Does Not Apply,Oversized Vehicle,NaN,NaN,NaN,d803d9ca816ccde76301f0bad3b8b5e5,Bus (Omnibus),Stake Truck,NaN,NaN,NaN


In [16]:
required_columns = ["latitude", "longitude"]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required geographical columns are missing: {missing_columns}"
    )

# Convert coordinates into numeric values
df["latitude"] = pd.to_numeric(
    df["latitude"], errors="coerce"
)

df["longitude"] = pd.to_numeric(
    df["longitude"], errors="coerce"
)

# Keep valid NYC geographical coordinates
valid_coordinates = (
    df["latitude"].between(40.4, 41.0)
    & df["longitude"].between(-74.3, -73.6)
)

geo_df = df.loc[valid_coordinates].copy()

geo_df = geo_df.dropna(
    subset=["latitude", "longitude"]
)

geo_df = geo_df.drop_duplicates(
    subset=["collision_id"]
) if "collision_id" in geo_df.columns else geo_df.drop_duplicates(
    subset=["latitude", "longitude"]
)

print("=" * 55)
print("GEOGRAPHICAL DATA VALIDATION")
print("=" * 55)

print("Total dataset records:", len(df))
print("Valid NYC geographical records:", len(geo_df))
print("Records excluded:", len(df) - len(geo_df))

print("\nFirst five valid geographical records:")

columns_to_display = [
    column for column in [
        "crash_date",
        "borough",
        "latitude",
        "longitude",
        "collision_id"
    ]
    if column in geo_df.columns
]

display(geo_df[columns_to_display].head())

GEOGRAPHICAL DATA VALIDATION
Total dataset records: 2301644
Valid NYC geographical records: 2052348
Records excluded: 249296

First five valid geographical records:


,crash_date,borough,latitude,longitude,collision_id
0,10/27/2026,MANHATTAN,40.720448,-74.005590,73fa239b448001d1b74e285a42d70488
1,10/27/2026,BROOKLYN,40.641815,-73.995810,1ca9703d3d16fae179d917b36065331a
2,10/27/2026,QUEENS,40.704185,-73.785980,005e53cae38738ee5bbedf0da83662f1
4,10/27/2026,MANHATTAN,40.860910,-73.921524,d803d9ca816ccde76301f0bad3b8b5e5
5,10/27/2026,BROOKLYN,40.637577,-73.925920,a6cb267c61821265bbda53bf0d007089


In [17]:
# Select a random sample for efficient clustering
SAMPLE_SIZE = min(100000, len(geo_df))

if SAMPLE_SIZE < 30:
    raise ValueError(
        "Not enough valid geographical records for DBSCAN."
    )

cluster_sample = geo_df.sample(
    n=SAMPLE_SIZE,
    random_state=42
).copy()

# Extract latitude and longitude
coordinates = cluster_sample[
    ["latitude", "longitude"]
].to_numpy()

# Convert coordinates from degrees to radians
coords_radians = np.radians(coordinates)

print("=" * 55)
print("DBSCAN DATA PREPARATION")
print("=" * 55)

print("Total NYC records:", len(geo_df))
print("Records selected for clustering:", len(cluster_sample))
print("Coordinate array shape:", coordinates.shape)
print("Radians array shape:", coords_radians.shape)

print("\nSample coordinate records:")

display(
    cluster_sample[
        ["latitude", "longitude"]
        + (["borough"] if "borough" in cluster_sample.columns else [])
    ].head()
)

print("\nDBSCAN input preparation completed successfully!")

DBSCAN DATA PREPARATION
Total NYC records: 2052348
Records selected for clustering: 100000
Coordinate array shape: (100000, 2)
Radians array shape: (100000, 2)

Sample coordinate records:


,latitude,longitude,borough
389268,40.603584,-74.018420,NaN
388331,40.880520,-73.857730,BRONX
1295765,40.825905,-73.879510,BRONX
1580193,40.664895,-73.875942,BROOKLYN
2094929,40.745707,-73.894728,NaN



DBSCAN input preparation completed successfully!


In [18]:
# DBSCAN configuration

EARTH_RADIUS_KM = 6371.0088

eps_km = 0.2
eps_radians = eps_km / EARTH_RADIUS_KM

min_samples = 30

dbscan_model = DBSCAN(
    eps=eps_radians,
    min_samples=min_samples,
    metric="haversine",
    algorithm="ball_tree",
    n_jobs=-1
)

print("=" * 55)
print("DBSCAN CONFIGURATION")
print("=" * 55)

print("Distance threshold:", eps_km, "km")
print("Minimum samples:", min_samples)
print("Distance metric: Haversine")
print("Algorithm: Ball Tree")
print("Records to analyze:", len(coords_radians))

DBSCAN CONFIGURATION
Distance threshold: 0.2 km
Minimum samples: 30
Distance metric: Haversine
Algorithm: Ball Tree
Records to analyze: 100000


In [19]:
# Perform clustering
cluster_labels = dbscan_model.fit_predict(
    coords_radians
)

# Store cluster labels
cluster_data = cluster_sample.copy()

cluster_data["cluster"] = cluster_labels

print("DBSCAN clustering completed successfully!")

print("\nCluster labels generated:")
print("Total records:", len(cluster_labels))
print("Unique labels:", np.unique(cluster_labels))

print("\nFirst five clustering results:")

display(
    cluster_data[
        ["latitude", "longitude", "cluster"]
    ].head()
)

DBSCAN clustering completed successfully!

Cluster labels generated:
Total records: 100000
Unique labels: [ -1   0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16
  17  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34
  35  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52
  53  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70
  71  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88
  89  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106
 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124
 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142
 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160
 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178
 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196
 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214
 215 216 217 218 2

,latitude,longitude,cluster
389268,40.603584,-74.018420,0
388331,40.880520,-73.857730,-1
1295765,40.825905,-73.879510,2
1580193,40.664895,-73.875942,-1
2094929,40.745707,-73.894728,1


In [20]:
# Count records in every cluster
cluster_counts = (
    cluster_data["cluster"]
    .value_counts()
    .sort_values(ascending=False)
)

# Exclude noise points (-1) when counting actual clusters
number_of_clusters = (
    cluster_data.loc[
        cluster_data["cluster"] != -1,
        "cluster"
    ].nunique()
)

noise_points = int(
    (cluster_data["cluster"] == -1).sum()
)

records_in_clusters = len(cluster_data) - noise_points

noise_percentage = (
    noise_points / len(cluster_data)
) * 100

print("=" * 55)
print("DBSCAN ACCIDENT HOTSPOT RESULTS")
print("=" * 55)

print("Records analyzed:", len(cluster_data))
print("Number of hotspot clusters:", number_of_clusters)
print("Records in clusters:", records_in_clusters)
print("Noise points:", noise_points)
print("Noise percentage:", round(noise_percentage, 2), "%")

print("\nCluster distribution:")

display(
    cluster_counts.rename("accident_count").to_frame().head(20)
)

DBSCAN ACCIDENT HOTSPOT RESULTS
Records analyzed: 100000
Number of hotspot clusters: 286
Records in clusters: 75152
Noise points: 24848
Noise percentage: 24.85 %

Cluster distribution:


,accident_count
cluster,
2,26895
-1,24848
5,15631
1,2404
24,1621
17,1087
3,1026
34,926
18,846


In [21]:
# Remove noise points before calculating hotspot statistics
hotspot_data = cluster_data[
    cluster_data["cluster"] != -1
].copy()

if hotspot_data.empty:
    raise ValueError(
        "DBSCAN did not detect any clusters. "
        "Review eps_km and min_samples."
    )

# Calculate hotspot statistics
hotspot_summary = (
    hotspot_data
    .groupby("cluster")
    .agg(
        accident_count=("cluster", "size"),
        center_latitude=("latitude", "mean"),
        center_longitude=("longitude", "mean"),
        latitude_spread=("latitude", "std"),
        longitude_spread=("longitude", "std")
    )
    .reset_index()
)

hotspot_summary[
    ["latitude_spread", "longitude_spread"]
] = hotspot_summary[
    ["latitude_spread", "longitude_spread"]
].fillna(0)

# Rank hotspots by accident frequency
hotspot_summary = hotspot_summary.sort_values(
    by="accident_count",
    ascending=False
).reset_index(drop=True)

hotspot_summary.insert(
    0,
    "hotspot_rank",
    range(1, len(hotspot_summary) + 1)
)

print("=" * 55)
print("ACCIDENT HOTSPOT RANKING")
print("=" * 55)

display(hotspot_summary.head(20))

# Save hotspot ranking
hotspot_summary.to_csv(
    REPORT_DIR / "hotspot_ranking.csv",
    index=False
)

print("\nHotspot ranking saved successfully!")

ACCIDENT HOTSPOT RANKING


,hotspot_rank,cluster,accident_count,center_latitude,center_longitude,latitude_spread,longitude_spread
0,1,2,26895,40.784870,-73.955423,0.045755,0.037723
1,2,5,15631,40.681111,-73.944371,0.021244,0.030422
2,3,1,2404,40.743516,-73.870273,0.008796,0.015861
3,4,24,1621,40.701877,-73.804211,0.007583,0.010973
4,5,17,1087,40.642877,-74.013880,0.008944,0.007184
5,6,3,1026,40.759835,-73.825615,0.004454,0.008685
6,7,34,926,40.821065,-73.888768,0.003763,0.009120
7,8,18,846,40.767083,-73.916379,0.003974,0.006335
8,9,44,635,40.582941,-73.958068,0.004379,0.006290
9,10,114,584,40.753732,-73.933492,0.003215,0.006824



Hotspot ranking saved successfully!


In [22]:
# Calculate map center
map_center = [
    cluster_data["latitude"].mean(),
    cluster_data["longitude"].mean()
]

accident_map = folium.Map(
    location=map_center,
    zoom_start=10,
    tiles="OpenStreetMap"
)

# Add a limited sample of accident locations to the map
map_sample = cluster_data.sample(
    n=min(5000, len(cluster_data)),
    random_state=42
)

for _, row in map_sample.iterrows():

    cluster_value = int(row["cluster"])

    if cluster_value == -1:
        color = "gray"
    else:
        color = "red"

    folium.CircleMarker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        radius=2,
        color=color,
        fill=True,
        fill_opacity=0.5,
        popup=f"Cluster: {cluster_value}"
    ).add_to(accident_map)

# Add hotspot center markers
for _, row in hotspot_summary.head(20).iterrows():

    popup_text = (
        f"Hotspot Rank: {int(row['hotspot_rank'])}<br>"
        f"Cluster: {int(row['cluster'])}<br>"
        f"Accidents: {int(row['accident_count'])}"
    )

    folium.Marker(
        location=[
            row["center_latitude"],
            row["center_longitude"]
        ],
        popup=folium.Popup(popup_text, max_width=300),
        tooltip=f"Hotspot Rank {int(row['hotspot_rank'])}",
        icon=folium.Icon(
            color="red",
            icon="info-sign"
        )
    ).add_to(accident_map)

map_path = MAP_DIR / "accident_hotspot_map.html"

accident_map.save(str(map_path))

print("Interactive hotspot map created successfully!")
print("Map saved at:", map_path)

display(accident_map)

Interactive hotspot map created successfully!
Map saved at: c:\Users\Pavan Naik\OneDrive\Desktop\DSV-project\visualizations\maps\accident_hotspot_map.html
